# Figure 4 and Figure S5: dCA1 task-state separation

Main panels **4d, 4f–h, 4j, 4l–n** and supplementary **S5a–f**. Historical source filenames use Fig.5 numbering.

This notebook follows **Xu et al., manuscript dated 30 October 2026**. Data loading, model fitting, panel selection and plotting are shown below as executable steps. Shared scientific methods remain in `src/`; this notebook does not delegate the figure to `run_figure()`.

Start with the separate manuscript source-data files listed in `data/required_files.csv`. These are analysis source tables and frozen map/embedding arrays, rather than raw movies. The aligned-data computation examples are in `00_recompute_metrics.ipynb`.

Run cells from top to bottom. `REFIT_STATISTICS=True` refits the manuscript models; set it to `False` to use the frozen estimates. Outputs are saved separately from inputs.

## 1. Paths and execution parameters

Set `PPRH_DATA_DIR` or edit `DATA_ROOT`. The default refits statistical models. No raw neural resampling is triggered here.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "main.py").is_file() and (p / "src/paper.py").is_file())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import SVG, Image, display
from paper_io import read_table, save_panel as _save_panel, write_tables, p_symbol, write_run_record

DATA_ROOT = Path(os.environ.get("PPRH_DATA_DIR", str(ROOT / "data/input")))
OUTPUT_ROOT = Path(os.environ.get("PPRH_OUTPUT_DIR", str(ROOT / "outputs/notebooks")))
REFIT_STATISTICS = os.environ.get("PPRH_REFIT_STATISTICS", "1") == "1"
FORMATS = ("svg", "png")
DPI = 300

data_root = DATA_ROOT
output = OUTPUT_ROOT / "Fig4"
refit, formats, dpi = REFIT_STATISTICS, FORMATS, DPI
output.mkdir(parents=True, exist_ok=True)
if not data_root.is_dir():
    raise FileNotFoundError("Supply the separate manuscript inputs through DATA_ROOT; see data/README.md.")


## 2. Analysis contract

Quantitative manifold results use 50 neurons, 15-dimensional SpectralEmbedding, 100 repeats, 100 label shuffles and `legacy_pooled` covariance. PV separation is `(mean_shuffle - observed_Spearman) / sd_shuffle`; manifold separation is `(observed_dprime - mean_shuffle) / sd_shuffle`. Models retain mouse, D1–D5 and movement covariates. Sample/choice and left/right contrasts remain distinct. The representative embeddings are three-dimensional visualizations (A4 D7 and C8 D5); the accepted A4 D7 example is outside the D1–D5 quantitative cohort window.

## 3. Notebook functions

The export function saves and previews each panel. Where needed, the source-table loader or annotation helper is defined here in full. Numerical estimators and shared plotting functions are imported in the relevant analysis cells.

In [ ]:
def save_panel(fig, output_root, panel, description, **kwargs):
    """Save one panel with the shared export method, then preview its result."""
    paths = _save_panel(fig, output_root, panel, description, **kwargs)
    svg = next((p for p in paths if p.endswith(".svg")), None)
    png = next((p for p in paths if p.endswith(".png")), None)
    if svg:
        display(SVG(filename=svg))
    elif png:
        display(Image(filename=png))
    return paths


In [ ]:
def _separation_bundle(data_root):
    from maze_analysis.fig5_state_separation import Fig5SourceBundle
    names = {'figure8_pv_bin': '01_Fig5_Figure8_high_dimensional_PV_spatial_bin_D1-D5.csv', 'figure8_pv_section': '02_Fig5_Figure8_high_dimensional_PV_sections_D1-D5.csv', 'figure8_manifold_bin': '03_Fig5_Figure8_neural_manifold_spatial_bin_D1-D5.csv', 'figure8_manifold_section': '04_Fig5_Figure8_neural_manifold_sections_D1-D5.csv', 'tmaze_pv_bin': '05_Fig5_Tmaze_high_dimensional_PV_spatial_bin_D1-D5.csv', 'tmaze_pv_section': '06_Fig5_Tmaze_high_dimensional_PV_sections_D1-D5.csv', 'tmaze_manifold_bin': '07_Fig5_Tmaze_neural_manifold_spatial_bin_D1-D5.csv', 'tmaze_manifold_section': '08_Fig5_Tmaze_neural_manifold_sections_D1-D5.csv'}
    return Fig5SourceBundle(**{k: read_table(data_root, 'fig5/source_data/' + v) for k, v in names.items()}, config_qc=pd.DataFrame())


## 4. Source data, statistics and panel-by-panel plots

Each panel cell names its source, contrast and display parameters. Use the current panel names below; historical filenames remain unchanged for traceability. Model fitting can take longer than figure rendering.

### Shared methods used below

In [ ]:
from figure_helpers import figure5 as h


### Shared methods used below

In [ ]:
from figure_helpers import figure5_heatmaps as hm


### Read the source data

In [ ]:
from maze_analysis.fig5_state_separation import build_fig5_section_summaries, select_panel_source

bundle = _separation_bundle(data_root)

summaries = build_fig5_section_summaries(bundle)

statistics = read_table(data_root, 'fig5/stats/Fig5_LMM_all_contrasts.csv')


In [ ]:
display(statistics.head())


### Fit the manuscript models or retain the frozen estimates

In [ ]:
if refit:
    result = h.fit_all_section_models(h.build_section_model_table(bundle), maxiter=1000)
    statistics = result.contrasts
    write_tables({'state_separation_refitted_estimands': statistics}, Path(output) / 'statistics')


In [ ]:
display(statistics.head())


### Prepare the next analysis step

In [ ]:
exports = []


### Fig4 f — prepare panel inputs

In [ ]:
spec, panel, table = (hm.PANEL_SPECS[0], 'Fig4_f', bundle.figure8_manifold_bin)

selected = table[table.contrast.eq(spec.contrast)].copy()


### Fig4 f — draw and export

In [ ]:
fig = hm.render_panel(selected, spec)

exports += save_panel(fig, output, panel, 'manifold_heatmap', old_label='D' if spec.source_family == 'figure8' else str(spec.accepted_tmaze_panel), formats=formats, dpi=dpi)


### FigS5 a — prepare panel inputs

In [ ]:
spec, panel, table = (hm.PANEL_SPECS[1], 'FigS5_a', bundle.tmaze_manifold_bin)

selected = table[table.contrast.eq(spec.contrast)].copy()


### FigS5 a — draw and export

In [ ]:
fig = hm.render_panel(selected, spec)

exports += save_panel(fig, output, panel, 'manifold_heatmap', old_label='D' if spec.source_family == 'figure8' else str(spec.accepted_tmaze_panel), formats=formats, dpi=dpi)


### FigS5 d — prepare panel inputs

In [ ]:
spec, panel, table = (hm.PANEL_SPECS[2], 'FigS5_d', bundle.tmaze_manifold_bin)

selected = table[table.contrast.eq(spec.contrast)].copy()


### FigS5 d — draw and export

In [ ]:
fig = hm.render_panel(selected, spec)

exports += save_panel(fig, output, panel, 'manifold_heatmap', old_label='D' if spec.source_family == 'figure8' else str(spec.accepted_tmaze_panel), formats=formats, dpi=dpi)


### Fig4 l — prepare panel inputs

In [ ]:
spec, panel, table = (hm.PANEL_SPECS[3], 'Fig4_l', bundle.tmaze_manifold_bin)

selected = table[table.contrast.eq(spec.contrast)].copy()


### Fig4 l — draw and export

In [ ]:
fig = hm.render_panel(selected, spec)

exports += save_panel(fig, output, panel, 'manifold_heatmap', old_label='D' if spec.source_family == 'figure8' else str(spec.accepted_tmaze_panel), formats=formats, dpi=dpi)


### Prepare the next analysis step

In [ ]:
specs = [('D', 'Fig4_g', 'figure8_manifold_section', 'left_vs_right_correct', ('start_arm', 'side_arm', 'reward_arm', 'end_arm'), h.plot_figure8_manifold_section_curves), ('F', 'Fig4_h', 'figure8_pv_section', 'left_vs_right_correct', ('start_arm', 'side_arm', 'reward_arm', 'end_arm'), h.plot_figure8_section_curves), ('O', 'Fig4_m', 'tmaze_manifold_section', 'sample_vs_choice', ('stem', 'choice_arm'), h.plot_tmaze_section_curves), ('R', 'Fig4_n', 'tmaze_pv_section', 'sample_vs_choice', ('stem', 'choice_arm'), h.plot_tmaze_section_curves), ('M', 'FigS5_b', 'tmaze_manifold_section', 'sample_left_vs_right', ('stem', 'choice_arm'), h.plot_tmaze_section_curves), ('P', 'FigS5_c', 'tmaze_pv_section', 'sample_left_vs_right', ('stem', 'choice_arm'), h.plot_tmaze_section_curves), ('N', 'FigS5_e', 'tmaze_manifold_section', 'choice_left_vs_right', ('stem', 'choice_arm'), h.plot_tmaze_section_curves), ('Q', 'FigS5_f', 'tmaze_pv_section', 'choice_left_vs_right', ('stem', 'choice_arm'), h.plot_tmaze_section_curves)]


### Fig4 g — prepare panel inputs

In [ ]:
old, panel, source_key, contrast, sections, plotter = ('D', 'Fig4_g', 'figure8_manifold_section', 'left_vs_right_correct', ('start_arm', 'side_arm', 'reward_arm', 'end_arm'), h.plot_figure8_manifold_section_curves)

observations = select_panel_source(bundle, old)

if source_key.startswith('figure8'):
    observations = observations[observations.section.isin(sections)].copy()
    contrast = h.FIGURE8_CONTRAST

summary = h._panel_summary(summaries, source_key=source_key, contrast=contrast, sections=sections)


In [ ]:
display(observations.head())


### Fig4 g — draw and export

In [ ]:
fig = plotter(h._display_sections(observations), summary, panel=old, stat_annotations=h._plot_annotations(statistics, observations))[0]

exports += save_panel(fig, output, panel, 'state_separation', old_label=old, formats=formats, dpi=dpi)


### Fig4 h — prepare panel inputs

In [ ]:
old, panel, source_key, contrast, sections, plotter = ('F', 'Fig4_h', 'figure8_pv_section', 'left_vs_right_correct', ('start_arm', 'side_arm', 'reward_arm', 'end_arm'), h.plot_figure8_section_curves)

observations = select_panel_source(bundle, old)

if source_key.startswith('figure8'):
    observations = observations[observations.section.isin(sections)].copy()
    contrast = h.FIGURE8_CONTRAST

summary = h._panel_summary(summaries, source_key=source_key, contrast=contrast, sections=sections)


In [ ]:
display(observations.head())


### Fig4 h — draw and export

In [ ]:
fig = plotter(h._display_sections(observations), summary, panel=old, stat_annotations=h._plot_annotations(statistics, observations))[0]

exports += save_panel(fig, output, panel, 'state_separation', old_label=old, formats=formats, dpi=dpi)


### Fig4 m — prepare panel inputs

In [ ]:
old, panel, source_key, contrast, sections, plotter = ('O', 'Fig4_m', 'tmaze_manifold_section', 'sample_vs_choice', ('stem', 'choice_arm'), h.plot_tmaze_section_curves)

observations = select_panel_source(bundle, old)

if source_key.startswith('figure8'):
    observations = observations[observations.section.isin(sections)].copy()
    contrast = h.FIGURE8_CONTRAST

summary = h._panel_summary(summaries, source_key=source_key, contrast=contrast, sections=sections)


In [ ]:
display(observations.head())


### Fig4 m — draw and export

In [ ]:
fig = plotter(h._display_sections(observations), summary, panel=old, stat_annotations=h._plot_annotations(statistics, observations))[0]

exports += save_panel(fig, output, panel, 'state_separation', old_label=old, formats=formats, dpi=dpi)


### Fig4 n — prepare panel inputs

In [ ]:
old, panel, source_key, contrast, sections, plotter = ('R', 'Fig4_n', 'tmaze_pv_section', 'sample_vs_choice', ('stem', 'choice_arm'), h.plot_tmaze_section_curves)

observations = select_panel_source(bundle, old)

if source_key.startswith('figure8'):
    observations = observations[observations.section.isin(sections)].copy()
    contrast = h.FIGURE8_CONTRAST

summary = h._panel_summary(summaries, source_key=source_key, contrast=contrast, sections=sections)


In [ ]:
display(observations.head())


### Fig4 n — draw and export

In [ ]:
fig = plotter(h._display_sections(observations), summary, panel=old, stat_annotations=h._plot_annotations(statistics, observations))[0]

exports += save_panel(fig, output, panel, 'state_separation', old_label=old, formats=formats, dpi=dpi)


### FigS5 b — prepare panel inputs

In [ ]:
old, panel, source_key, contrast, sections, plotter = ('M', 'FigS5_b', 'tmaze_manifold_section', 'sample_left_vs_right', ('stem', 'choice_arm'), h.plot_tmaze_section_curves)

observations = select_panel_source(bundle, old)

if source_key.startswith('figure8'):
    observations = observations[observations.section.isin(sections)].copy()
    contrast = h.FIGURE8_CONTRAST

summary = h._panel_summary(summaries, source_key=source_key, contrast=contrast, sections=sections)


In [ ]:
display(observations.head())


### FigS5 b — draw and export

In [ ]:
fig = plotter(h._display_sections(observations), summary, panel=old, stat_annotations=h._plot_annotations(statistics, observations))[0]

exports += save_panel(fig, output, panel, 'state_separation', old_label=old, formats=formats, dpi=dpi)


### FigS5 c — prepare panel inputs

In [ ]:
old, panel, source_key, contrast, sections, plotter = ('P', 'FigS5_c', 'tmaze_pv_section', 'sample_left_vs_right', ('stem', 'choice_arm'), h.plot_tmaze_section_curves)

observations = select_panel_source(bundle, old)

if source_key.startswith('figure8'):
    observations = observations[observations.section.isin(sections)].copy()
    contrast = h.FIGURE8_CONTRAST

summary = h._panel_summary(summaries, source_key=source_key, contrast=contrast, sections=sections)


In [ ]:
display(observations.head())


### FigS5 c — draw and export

In [ ]:
fig = plotter(h._display_sections(observations), summary, panel=old, stat_annotations=h._plot_annotations(statistics, observations))[0]

exports += save_panel(fig, output, panel, 'state_separation', old_label=old, formats=formats, dpi=dpi)


### FigS5 e — prepare panel inputs

In [ ]:
old, panel, source_key, contrast, sections, plotter = ('N', 'FigS5_e', 'tmaze_manifold_section', 'choice_left_vs_right', ('stem', 'choice_arm'), h.plot_tmaze_section_curves)

observations = select_panel_source(bundle, old)

if source_key.startswith('figure8'):
    observations = observations[observations.section.isin(sections)].copy()
    contrast = h.FIGURE8_CONTRAST

summary = h._panel_summary(summaries, source_key=source_key, contrast=contrast, sections=sections)


In [ ]:
display(observations.head())


### FigS5 e — draw and export

In [ ]:
fig = plotter(h._display_sections(observations), summary, panel=old, stat_annotations=h._plot_annotations(statistics, observations))[0]

exports += save_panel(fig, output, panel, 'state_separation', old_label=old, formats=formats, dpi=dpi)


### FigS5 f — prepare panel inputs

In [ ]:
old, panel, source_key, contrast, sections, plotter = ('Q', 'FigS5_f', 'tmaze_pv_section', 'choice_left_vs_right', ('stem', 'choice_arm'), h.plot_tmaze_section_curves)

observations = select_panel_source(bundle, old)

if source_key.startswith('figure8'):
    observations = observations[observations.section.isin(sections)].copy()
    contrast = h.FIGURE8_CONTRAST

summary = h._panel_summary(summaries, source_key=source_key, contrast=contrast, sections=sections)


In [ ]:
display(observations.head())


### FigS5 f — draw and export

In [ ]:
fig = plotter(h._display_sections(observations), summary, panel=old, stat_annotations=h._plot_annotations(statistics, observations))[0]

exports += save_panel(fig, output, panel, 'state_separation', old_label=old, formats=formats, dpi=dpi)


### Shared methods used below

In [ ]:
from maze_analysis import manifold_trajectory as trajectory


### Fig4 d — prepare panel inputs

In [ ]:
from figure_helpers import tmaze_trajectory as phase_trajectory

session = Path(data_root) / 'representatives/figure8_A4_D7'

points = trajectory.load_correct_left_right_point_session(session)

means = trajectory.load_correct_left_right_trajectory_session(session, n_bins=56)


### Fig4 d — draw and export

In [ ]:
fig = trajectory.plot_correct_left_right_points(points, view_elev=75, view_azim=-90)

exports += save_panel(fig, output, 'Fig4_d', 'Figure8_latent_points_A4_D7', formats=formats, dpi=dpi)


### Fig4 d — draw and export

In [ ]:
fig = trajectory.plot_correct_left_right_trajectories(means, view_elev=75, view_azim=-90)

exports += save_panel(fig, output, 'Fig4_d', 'Figure8_mean_trajectories_A4_D7', formats=formats, dpi=dpi)


### Fig4 j — prepare panel inputs

In [ ]:
latent = read_table(data_root, 'representatives/tmaze_phase_latent_points.csv')

conditions = {phase: {'points': rows[['spectral_1', 'spectral_2', 'spectral_3']].to_numpy(), 'bins': rows.spatial_bin.to_numpy(), 'trial_ids': rows.phase_trial_index.to_numpy()} for phase, rows in latent.groupby('condition', sort=False)}

means, _ = phase_trajectory.mean_trajectories(conditions)


### Fig4 j — draw and export

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(2.05, 3.45), gridspec_kw={'height_ratios': [1, 0.78], 'hspace': 0.03})

max_bin = int(latent.spatial_bin.max())

phase_trajectory.draw_point_cloud(axes[0], conditions, max_bin)

phase_trajectory.draw_mean_trajectories(axes[1], conditions, means, max_bin)

fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)

exports += save_panel(fig, output, 'Fig4_j', 'Tmaze_phase_latent_trajectories', formats=formats, dpi=dpi)


## 5. Export record

Panel files and any refitted model tables are saved under the figure output directory. The record identifies the selected manuscript workflow and whether statistics were refitted.

In [ ]:
write_run_record(output / "run.json", figure="4", refit=refit, exports=exports)
display(pd.DataFrame({"exported_file": [str(Path(p).relative_to(output)) for p in exports]}))
